In [1]:
import numpy as np
from torchvision.models import vgg11_bn,VGG11_BN_Weights 
import torch.nn as nn
from torch.utils.data import DataLoader
from scipy.special import lambertw
from sklearn.cluster import KMeans
import torch
import torch.nn as nn
import torch.optim as optim
import copy
import torch
import numpy as np
import random
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10
from torchvision import transforms
from sklearn.metrics import confusion_matrix
import torch.nn.functional as F
import os
import pandas as pd


In [2]:
class PretrainedVGG11(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.5):
        super(PretrainedVGG11, self).__init__()

        self.conv_block1 = self._make_conv_block(3, 64, 2)
        self.conv_block2 = self._make_conv_block(64, 128, 2)
        self.conv_block3 = self._make_conv_block(128, 256, 3)
        self.conv_block4 = self._make_conv_block(256, 512, 3)
        self.conv_block5 = self._make_conv_block(512, 512, 3)

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        )
    def _make_conv_block(self, in_channels, out_channels, num_convs):
        layers = []
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1))
        layers.append(nn.BatchNorm2d(out_channels))  # BatchNorm after the convolution
        layers.append(nn.ReLU())
        for _ in range(num_convs - 1):
            layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1))
            layers.append(nn.BatchNorm2d(out_channels))  # BatchNorm after the convolution
            layers.append(nn.ReLU())
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        x = self.conv_block5(x)
        x = self.avg_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


In [3]:

# class PretrainedVGG11(nn.Module):
#     def __init__(self, num_classes):
#         super(PretrainedVGG11, self).__init__()
#         # Carica il modello VGG16 pre-addestrato
#         self.vgg = vgg11_bn(weights=VGG11_BN_Weights.DEFAULT)
#         for param in self.vgg.parameters():
#             param.requires_grad = False
#         self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
#         # Modifica l'ultimo strato della rete per il dataset CIFAR-10

#         self.vgg.classifier[6] = nn.Linear(4096, num_classes)

#     def forward(self, x):
#         return self.vgg(x)

In [4]:


def calculate_confidence_score(dataloader, num_classes, lambda_reg, device,client_id,agent_wights,server_classes,round_federated):
    # print(f"calcolo confidenze agente {client_id}")
 
    global_model_ = PretrainedVGG11(num_classes=server_classes)
    
    global_model_.load_state_dict(agent_wights)
    criterion = nn.CrossEntropyLoss(reduction='none')
    global_model_.eval()
    log_C = np.log(num_classes)
    #lambda regularization base = 0.5
    missing_class_agents = server_classes-num_classes
    # if missing_class_agents >0:
    #     lambda_reg-= round(missing_class_agents/10,10)
    #     print(f"agente {client_id} reweight regularization missing {missing_class_agents} -> {lambda_reg}")        
    total_sigma = 0.0
    num_samples = 0
    CM = np.zeros((num_classes, num_classes), dtype=int)
    max_print = 0
    global_model_.to(device)

    with torch.no_grad():
        for data in dataloader:
            # print(f"num_samples {num_samples}")
            images, labels = data
            images = images.to(device)
            labels = labels.to(device)
            # Predizioni del modello
            outputs = global_model_(images)
            # probabilities = torch.softmax(outputs, dim=1)
            # losses = -torch.log(probabilities[range(len(labels)), labels])
            # _, predicted = torch.max(outputs.data, 1)
            _, preds = torch.max(outputs, 1)           
            CM+=confusion_matrix(labels.cpu().numpy(), preds.cpu().numpy(),labels=range(num_classes))
            
            losses = criterion(outputs, labels)
            # print(losses)
            for loss_val in losses:
                loss = loss_val.item()
                # Calcola la confidenza
                
                max_term = max(-2 / np.exp(1), (loss - log_C) / lambda_reg)
               
                    # print(f"loss - logC / lambda reg -> {(loss - log_C) / lambda_reg}")
                    # print(f" -2 / np.exp(1) -> {-2 / np.exp(1)}")
                    # print(f"max term {max_term}")
                   
                W_input = round(0.5 * max_term,15)
                W_output = lambertw(W_input).real
                sigma_j = np.exp(-W_output)
                total_sigma += sigma_j
                num_samples += 1
                # if max_print <=5:
                #     max_print+=1
                #     print(f"w input {W_input}")
                #     print(f"w out {W_output}")
                #     print(f"sigma_j {sigma_j}")
            # del images, labels, outputs
            # torch.cuda.empty_cache()
    # print(f"total_sigma {total_sigma}")
    # print(f"num_samples {num_samples}")
    confidenza_media = total_sigma / num_samples if num_samples > 0 else 0
    # print(f"confidenza_media agente {client_id}: {confidenza_media}")
    pd.DataFrame(CM, index=[str(i) for i in range(num_classes)], columns=[str(i) for i in range(num_classes)]).to_csv(f"""./matrici_confusione/round_{round_federated}_agente_{client_id}.csv""")
    # del global_model_
    return confidenza_media

In [5]:

def classify_clients_by_confidence(normalized_confidences):
 
    # Converti le confidenze normalizzate in una lista ordinata
    client_ids = list(normalized_confidences.keys())
    confidences = np.array(list(normalized_confidences.values())).reshape(-1, 1)

    # Applica k-means clustering con k=2
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    kmeans.fit(confidences)
    centroids = kmeans.cluster_centers_.flatten()

    # Ordina i centroidi (µlower e µupper)
    mu_lower, mu_upper = np.min(centroids), np.max(centroids)

    # Classifica i clienti
    honest_clients = set()
    malicious_clients = set()

    for client_id, sigma_i in normalized_confidences.items():
        # Classifica in base alla distanza dai centroidi
        if abs(sigma_i - mu_upper) < abs(sigma_i - mu_lower):
            honest_clients.add(client_id)
        else:
            malicious_clients.add(client_id)

    return honest_clients, malicious_clients


In [6]:


def federated_training_with_confidence(client_loaders, server_loader, num_agents=5, epochs=5, num_rounds=10, device='cuda',agent_classes=None,
                                       server_classes=10,lambda_reg = 0.25,resume_from_checkpoint = False,check_point_name = 'checkpoint.pth.tar',round_resuming = None):
    """
    Addestramento federato con calcolo delle confidenze e identificazione dei client onesti.

    Args:
    - client_loaders: Lista di DataLoader per ciascun client.
    - server_loader: DataLoader per il server.
    - num_agents: Numero di client.
    - epochs: Numero di epoche per l'addestramento locale.
    - num_rounds: Numero di round di addestramento federato.
    - device: Dispositivo di esecuzione ('cpu' o 'cuda').

    Returns:
    - agents_confidence: Confidenze calcolate per ogni client.
    - aggregated_weights: Pesi aggregati del modello globale.
    """

    
    rounds_agent_confidences = {}

    # Modello globale
    # global_model = PretrainedVGG11(num_classes=server_classes).to(device)
    # global_weights = global_model.state_dict()
    already_resumed_from_checkpoint = False
    decresase_round = 0
    start_round = round_resuming if round_resuming else 0

    
    for round_num in range(start_round,num_rounds):
        
            
        print(f"\nRound {round_num + 1}/{num_rounds} iniziato...")

        agent_weights = {}
        data_lengths = {}
        clip_value = 5
        first_round = True
        # Addestramento locale
        for client_id, loader in enumerate(client_loaders):
            if not first_round:
                del local_model,optimizer,checkpoint,criterion
            print(f" Agente {client_id} inizia l'addestramento")
            # print(f"numero classi agente {client_id}: {agent_classes[client_id]}")
            local_model = PretrainedVGG11(num_classes=server_classes)
            # local_model.load_state_dict(global_weights)
            # local_model= nn.DataParallel(local_model)

            if resume_from_checkpoint and not already_resumed_from_checkpoint:
                checkpoint = load_checkpoint(check_point_name)
                local_model.load_state_dict(checkpoint['state_dict'])
                # optimizer.load_state_dict(checkpoint['optimizer'])
                print(f"Resuming from checkpoint agent {client_id}")
            elif not first_round:
                print("loading global weights")
                local_model.load_state_dict(global_weights)
                
            optimizer = optim.SGD(local_model.parameters(), lr=0.01,weight_decay = 1e-4, momentum=0.9)
            criterion = nn.CrossEntropyLoss()
            local_model.to(device)
            for epoch in range(epochs):
                print(f" Epoca {epoch} di {epochs}")

                local_model.train()
                for images, labels in loader:
                    images, labels = images.to(device), labels.to(device)
                    optimizer.zero_grad()
                    outputs = local_model(images)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    # torch.nn.utils.clip_grad_norm_(local_model.parameters(), clip_value)
                    optimizer.step()
                    # del images, labels, outputs
                    # torch.cuda.empty_cache()
           
            # print(local_model.state_dict())
            agent_weights[client_id]=local_model.state_dict()
            data_lengths[client_id] = len(loader.dataset)
            # del local_model,optimizer,criterion
        # Calcolo delle confidenze
        agent_confidences = {
            client_id: calculate_confidence_score(server_loader, agent_classes[client_id], lambda_reg, 
                                                  device,client_id,agent_weights[client_id],server_classes,round_num)
            for client_id in range(num_agents)
        }

        print(f"agents confidences {agent_confidences}")
     # Rescaling Min-Max delle confidenze
        conf_values = list(agent_confidences.values())
        min_conf = min(conf_values)
        max_conf = max(conf_values)
    
        if max_conf > min_conf:
            print("max > min conf")
            normalized_confidences = {
                client_id: (conf - min_conf) / (max_conf - min_conf)
                for client_id, conf in agent_confidences.items()
            }
        else:
            print("max <= min conf")
            normalized_confidences = {client_id: 1.0 for client_id in agent_confidences}

        print(f"Confidenze Normalizzate (Min-Max): {normalized_confidences}")
        rounds_agent_confidences[round_num + 1] = normalized_confidences

        # Classifica i client
        honest_clients, malicious_clients = classify_clients_by_confidence(normalized_confidences)
        print(f"Clienti onesti: {honest_clients}")
        print(f"Clienti malevoli: {malicious_clients}")
        # return agent_weights, data_lengths, normalized_confidences, honest_clients
        # Aggrega i pesi
        global_weights = reweighted_aggregation(agent_weights, data_lengths, normalized_confidences, honest_clients)
        # global_model.load_state_dict(global_weights)
        already_resumed_from_checkpoint = True
        print(f"Checkpoint model round {round_num +1}")
        checkpoint = {
            'state_dict':global_weights
            # 'optimizer': optimizer.state_dict(),
        }
        save_checkpoint(checkpoint,f"./checkpoint/{round_num + 1}_checkpoint.pth.tar")
        save_logs(honest_clients, malicious_clients,round_num+1,normalized_confidences,agent_confidences)
        print(f"Round {round_num + 1} completato.")
        decresase_round +=1
        first_round = False
        # if epochs >=2 and decresase_round == 2:
        #     epochs-=1
        #     decresase_round = 0
    return rounds_agent_confidences, global_weights


In [7]:
import os
import pandas as pd
import json

def save_logs(honest_clients, malicious_clients, round_num, normalized_confidences, agent_confidences, log_file='logs.csv'):
    log_file = f'./logs/{round_num}_logs.csv'
    # Verifica se il file di log esiste
    file_exists = os.path.isfile(log_file)
    
    # Crea un dizionario con i dati da salvare
    log_data = {
        'Round': round_num,
        'Honest_Clients': json.dumps(list(honest_clients)),
        'Malicious_Clients': json.dumps(list(malicious_clients)),
        'Normalized_Confidences': json.dumps(normalized_confidences),
        'Agent_Confidences': json.dumps(agent_confidences)
    }
    
    # Crea un DataFrame
    df = pd.DataFrame([log_data])
    
    # Scrivi o aggiungi al file CSV
    if not file_exists:
        # Scrivi con intestazioni
        df.to_csv(log_file, index=False)
        print(f"File di log creato e dati del round {round_num} salvati.")
    else:
        # Aggiungi senza intestazioni
        df.to_csv(log_file, mode='a', header=False, index=False)
        print(f"Dati del round {round_num} aggiunti al file di log esistente.")


In [8]:

def reweighted_aggregation(
    agent_weights: dict,
    data_lengths: dict,
    confidences: dict,
    honest_clients: list or set
):
    

    # --- 1. Calcolo dei pesi originali w_orig,i basati su data_lengths ---
    sum_data_lengths = sum(data_lengths[cid] for cid in honest_clients)
    if sum_data_lengths == 0:
        raise ValueError("La somma delle lunghezze di tutti i client onesti è 0!")

    w_orig = {}
    for cid in honest_clients:
        w_orig[cid] = data_lengths[cid] / sum_data_lengths

    # --- 2. Normalizzazione delle confidenze per ottenere r_norm,i ---
    sum_confidences = sum(confidences[cid] for cid in honest_clients)
    if sum_confidences == 0:
        
        print("Attenzione: la somma delle confidenze è 0.")
        r_norm = {cid: 1.0 / len(honest_clients) for cid in honest_clients}
    else:
        r_norm = {}
        for cid in honest_clients:
            r_norm[cid] = confidences[cid] / sum_confidences

    # --- 3. Calcolo dei pesi finali w_final,i ---
    w_final = {}
    for cid in honest_clients:
        w_final[cid] = r_norm[cid] * w_orig[cid]

    # --- 4. Aggregazione dei pesi ---
    
    numerator_sd = None
    denominator = 0.0

    for idx, cid in enumerate(honest_clients):
        local_sd = agent_weights[cid]  # state_dict del client i
        weight_factor = w_final[cid] * data_lengths[cid]
        denominator += weight_factor

        # Se è il primo client onesto, crea una copia profonda e scala
        if numerator_sd is None:
            numerator_sd = copy.deepcopy(local_sd)
            for key in numerator_sd:
                numerator_sd[key] = numerator_sd[key] * weight_factor
        else:
            # Somma pesata in tutti i layer
            for key in local_sd:
                numerator_sd[key] += local_sd[key] * weight_factor

    if denominator == 0:
        print("Attenzione: denominator = 0 durante l'aggregazione. Uso denominator=1 di default.")
        denominator = 1.0

    # Dividiamo per la somma dei pesi
    for key in numerator_sd:
        numerator_sd[key] /= denominator

     
    return numerator_sd


In [9]:


def make_federated_data(    dataset,    num_clients=5,    alpha=0.5,    server_fraction=0.1,    batch_size=32,    shuffle=True,    seed=42,
                       save_path = None,num_workers=4):
 
    np.random.seed(seed)
    random.seed(seed)

    total_len = len(dataset)
    num_classes = 10  # se CIFAR-10
    os.makedirs(save_path, exist_ok=True)

    server_indices_path = os.path.join(save_path, 'server_indices.pt')
    client_indices_paths = [os.path.join(save_path, f'client_{i}_indices.pt') for i in range(num_clients)]
    
    indices_exist = os.path.exists(server_indices_path) and all(os.path.exists(path) for path in client_indices_paths)

    # ------------------------------------------------
    # 1) Selezioniamo i campioni per il server
    # ------------------------------------------------
    server_size = int(server_fraction * total_len)
    all_indices = list(range(total_len))
    
    if indices_exist:
        # Carica gli indici salvati
        server_indices = torch.load(server_indices_path)
        client_indices = []
        for path in client_indices_paths:
            client_indices.append(torch.load(path))
        print("[Info] Indici caricati dai file di split.")
    else:
        random.shuffle(all_indices)
        
        server_indices = all_indices[:server_size]
        client_indices_all = all_indices[server_size:]
        torch.save(server_indices, server_indices_path)
        print(f"[Info] Indici del server salvati in {server_indices_path}.")

        # ------------------------------------------------
        # 2) Raggruppiamo i rimanenti per classe
        # ------------------------------------------------
        indices_by_class = [[] for _ in range(num_classes)]
        for idx in client_indices_all:
            _, label = dataset[idx]
            indices_by_class[label].append(idx)

        # ------------------------------------------------
        # 3) Inizializziamo array per i client
        # ------------------------------------------------
        client_indices = [[] for _ in range(num_clients)]
    
        # Per tracciare la distribuzione {client: {classe: count}}
        agent_class_dist = {cid: {c: 0 for c in range(num_classes)} 
                            for cid in range(num_clients)}

        # ------------------------------------------------
        # 4) Partizione Dirichlet su ogni classe
        # ------------------------------------------------
        for c in range(num_classes):
            idxs_c = indices_by_class[c]
            random.shuffle(idxs_c)
            n_class = len(idxs_c)
            if n_class == 0:
                continue
    
            # estraiamo p ~ Dirichlet(alpha)
            proportions = np.random.dirichlet([alpha]*num_clients)
    
            # conteggi
            class_counts = (proportions * n_class).astype(int)
    
            gap = n_class - np.sum(class_counts)
            # distribuiamo l'eventuale gap
            while gap > 0:
                i = random.randint(0, num_clients-1)
                class_counts[i] += 1
                gap -= 1
    
            # Assegniamo i campioni di classe c
            start = 0
            for i in range(num_clients):
                count = class_counts[i]
                assigned_indices = idxs_c[start : start + count]
                client_indices[i].extend(assigned_indices)
                # Aggiorniamo la distribuzione
                agent_class_dist[i][c] += count
    
                start += count
        # Salva gli indici per ogni client
        for i in range(num_clients):
            torch.save(client_indices[i], client_indices_paths[i])
            print(f"[Info] Indici del client {i} salvati in {client_indices_paths[i]}.")
    # ------------------------------------------------
    # 5) Creiamo i DataLoader per i client
    # ------------------------------------------------
    client_loaders = []
    agent_classe = {}

    for i in range(num_clients):
        subset_i = Subset(dataset, client_indices[i])
        loader_i = DataLoader(subset_i, batch_size=batch_size, shuffle=shuffle,num_workers=num_workers)
        client_loaders.append(loader_i)
        class_counts = torch.zeros(num_classes, dtype=torch.long)

        total_per_client = len(subset_i)
        for _, labels in loader_i:
            # Conta le occorrenze di ciascuna classe nel batch
            batch_counts = torch.bincount(labels, minlength=num_classes)
            # Aggiungi i conteggi del batch al totale
            class_counts += batch_counts
        
        # Converti i conteggi in un dizionario
        distribution = {cls: int(count) for cls, count in enumerate(class_counts)}
        
        #approccio 1: ogni agente riceve sempre 10
        # agent_classe[i]=len(set(distribution.keys()))

        #approccio 2: ogni agente riceve il numero effettivo di classi, dove le istanze sono >0
        conta_classi = 0
        for key in distribution.keys():
            if distribution[key] >0:
                conta_classi+=1
        print(f"""[Client {i}] #campioni={total_per_client} -> Distribuzione classi:
        {distribution}""")
        agent_classe[i] = conta_classi
        

    server_subset = Subset(dataset, server_indices)
    server_loader = DataLoader(
        server_subset, batch_size=batch_size, shuffle=True,num_workers = num_workers
    )
    print(f"[Server] #campioni = {len(server_subset)} (fraction={server_fraction:.2f})")
    
    return client_loaders, server_loader, agent_classe


In [10]:
def save_checkpoint(state, filename="checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

In [11]:
def load_checkpoint(filename):
    print("=> Loading checkpoint")
    return  torch.load(filename)
   

In [12]:
test_case = 'base'
save_path = './dataloaders_indices'
num_clients = 25
server_data_fraction = 0.2 
batch_size = 32
alpha = 0.5
epochs = 20
num_rounds = 200
server_classes = 10
lambda_reg = 0.5
resume_from_checkpoint = False
check_point_name = './checkpoint/99_checkpoint.pth.tar',
round_resuming = None #none default

In [13]:
# transform = transforms.Compose([
     
#     transforms.ToTensor(),
#     transforms.Normalize(  mean=(0.49139968, 0.48215841, 0.44653091),
#     std=(0.24703223, 0.24348513, 0.26158784))
# ])
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(size=32, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = CIFAR10(root='./datatest_training', train=True, download=True, transform=transform)
client_loaders, server_loader,agent_classes = make_federated_data(
        dataset=dataset,
        num_clients=num_clients,
        alpha=alpha,
        server_fraction=server_data_fraction,
        batch_size=batch_size,
        save_path = save_path
    )

test_dataset =   CIFAR10(root='./datatest_test', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)


Files already downloaded and verified
[Info] Indici caricati dai file di split.


/tmp/ipykernel_1436547/1637121963.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  server_indices = torch.load(server_indices_path)
/tmp/ipykernel_1436547/1637121963.py:

[Client 0] #campioni=2227 -> Distribuzione classi:
        {0: 47, 1: 709, 2: 48, 3: 152, 4: 92, 5: 284, 6: 1, 7: 280, 8: 284, 9: 330}
[Client 1] #campioni=1475 -> Distribuzione classi:
        {0: 224, 1: 108, 2: 15, 3: 2, 4: 1, 5: 180, 6: 347, 7: 53, 8: 62, 9: 483}
[Client 2] #campioni=767 -> Distribuzione classi:
        {0: 9, 1: 2, 2: 24, 3: 37, 4: 13, 5: 494, 6: 67, 7: 46, 8: 5, 9: 70}
[Client 3] #campioni=1921 -> Distribuzione classi:
        {0: 2, 1: 1, 2: 634, 3: 1, 4: 312, 5: 67, 6: 8, 7: 496, 8: 0, 9: 400}
[Client 4] #campioni=913 -> Distribuzione classi:
        {0: 128, 1: 44, 2: 125, 3: 93, 4: 173, 5: 25, 6: 43, 7: 280, 8: 1, 9: 1}
[Client 5] #campioni=885 -> Distribuzione classi:
        {0: 1, 1: 313, 2: 272, 3: 160, 4: 65, 5: 0, 6: 39, 7: 4, 8: 5, 9: 26}
[Client 6] #campioni=1864 -> Distribuzione classi:
        {0: 371, 1: 23, 2: 465, 3: 194, 4: 8, 5: 323, 6: 71, 7: 99, 8: 245, 9: 65}
[Client 7] #campioni=1269 -> Distribuzione classi:
        {0: 11, 1: 6, 2: 276, 3:

In [17]:
from collections import Counter
import torch

# Supponiamo che il DataLoader del server sia server_loader
class_counts = Counter()

# Itera sul DataLoader
for images, labels in server_loader:
    labels = labels.numpy() if isinstance(labels, torch.Tensor) else labels
    class_counts.update(labels)

# Stampa la distribuzione delle classi
print("Distribuzione delle classi nel Server Set:", class_counts.order())


AttributeError: 'Counter' object has no attribute 'order'

In [14]:
agents_confidence, aggregated_weights = federated_training_with_confidence(
    client_loaders=client_loaders,
    server_loader=server_loader,
    num_agents=num_clients,
    epochs=epochs,
    num_rounds=num_rounds,
    device='cuda',
    agent_classes = agent_classes,
    server_classes = server_classes,
    lambda_reg = lambda_reg,
    resume_from_checkpoint = resume_from_checkpoint,
    check_point_name = check_point_name,
    round_resuming = round_resuming
)


Round 1/200 iniziato...
 Agente 0 inizia l'addestramento
 Epoca 0 di 20


KeyboardInterrupt: 

In [ ]:
last_checkpoint_path = './checkpoint/200_checkpoint.pth.tar'

In [ ]:
check_point = load_checkpoint(last_checkpoint_path)
check_point

In [ ]:
device = 'cuda'
num_classes = 10
global_model = PretrainedVGG11(num_classes=10).to('cuda')

global_model.load_state_dict(check_point["state_dict"])
criterion = nn.CrossEntropyLoss(reduction='none')
global_model.eval()
global_model.to('cuda')
# log_C = np.log(num_classes)

total_sigma = 0.0
num_samples = 0
CM = np.zeros((10, 10), dtype=int)

with torch.no_grad():
    for data in test_loader:
        # print(f"num_samples {num_samples}")
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        # Predizioni del modello
        outputs = global_model(images)
        probabilities = torch.softmax(outputs, dim=1)
        print(probabilities)
        # losses = -torch.log(probabilities[range(len(labels)), labels])
        # _, predicted = torch.max(outputs.data, 1)
        _, preds = torch.max(outputs, 1)           
        CM+=confusion_matrix(labels.cpu().numpy(), preds.cpu().numpy(),labels=range(num_classes))

In [ ]:
pd.DataFrame(CM)

In [ ]:
cm = CM

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
y_true = []
y_pred = []
num_classes = cm.shape[0]
for true_label in range(num_classes):
    for pred_label in range(num_classes):
        count = cm[true_label, pred_label]
        y_true.extend([true_label] * count)
        y_pred.extend([pred_label] * count)


accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=range(num_classes), average='weighted', zero_division=0
)

In [ ]:
precision, recall, f1,accuracy